In [30]:
import sys
import importlib
import pathlib
import numpy as np
import matplotlib.pyplot as plt

# 1. Ensure Python can find your emulator library
source_dir = pathlib.Path.cwd().parent / 'software' / 'source'
if str(source_dir) not in sys.path:
    sys.path.append(str(source_dir))

import qick_emu
importlib.reload(qick_emu)
from qick_emu import QickEmu
from qick.asm_v2 import AveragerProgramV2

## Overview: real-time feedback in QICK

The following demo shows an experiment that demonstrates the QICK tprocv2 capabilities, namely the arithmetic logic capabilities, branching logic, real-time parametric pulse generation, and LFSR based random number generation. The configuration is as follows: channel 0 of the RFSoC is plugged into readout channel 4. A random number is generated with the top 16 bits shifted into a register. The random number is mapped to the amplitude wave parameter of that pulse envelope. The pulse is sent into channel 4, where it is demodulated and averaged, returning an amplitude. The program checks the measured amplitude. If the amplitude is lower than a given amount, it fires a pulse from the channel 2 DAC. TRM manual describing instructions found here: https://github.com/meeg/qick_demos_sho/blob/main/tprocv2/qick_processor_TRM.pdf

In [31]:
# 2. Define the path to your local hardware configuration
REPO_ROOT = pathlib.Path.cwd().parent
CFG_PATH = REPO_ROOT / "notebooks" / "qick_config_216.json"

# 3. Instantiate the Emulator (Replaces the Pyro4 network connection)
emu = QickEmu(str(CFG_PATH))

# 4. Extract the QickConfig object (Replaces soc.get_cfg())
soccfg = emu.soccfg

# Print the configuration to verify it loaded correctly
print(soccfg)

QICK library version mismatch: 0.2.366 remote (the board), 0.2.370 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


QICK running on ZCU216, software version 0.2.366

Firmware configuration (built Sat Sep 28 22:15:40 2024):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc timing clock, DAC tile 1, DAC tile 2, DAC tile 3], [DAC tile 0], [ADC tile 2]

	16 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 65536 complex samples (6.838 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0
	1:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 16384 complex samples (1.709 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 0, blk 1 is 1_228 on JHC2, or QICK box DAC port 1
	2:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 32768 complex samples (3.419 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 0, blk 2 is 2_228 on JHC1, or QICK box DAC port 2
	3:	axis_signal_gen_v6 - fs=9584.640 Msps, fabri

In [32]:
from qick.tprocv2_assembler import Assembler

def pstr2asm_inst(self, pstr_lst):
    pstr = '\n'.join(pstr_lst) + '\n'
    asm_dict, labels = Assembler.str_asm2list(pstr)
    for label, addr in labels.items():
        if addr != label:   # skip pre-defined assembler symbols (e.g. 's15': 's15')
            self.label(label)
    for instr in asm_dict:
        self.asm_inst(instr)

# -----------------------------------------------------------------------------
# FIXED MACROS: Inject raw dictionaries to bypass isolated label parsing errors
# -----------------------------------------------------------------------------
def NOP(self):
    self.label('END_SETUP')

def JUMP(self, label):
    self.asm_inst({'CMD': 'JUMP', 'LABEL': label})

def CALL(self, label, cond=None):
    inst = {'CMD': 'CALL', 'LABEL': label}
    if cond is not None:
        inst['IF'] = cond
    self.asm_inst(inst)

def exitREP(self):
    self.label('exitREP')

def SETUP_PULSE(self, cfg): 
    pstr_lst = [
        f'REG_WR w_gain imm #{str(cfg["gain"])}',
        f'REG_WR w_phase imm #{self.deg2reg(cfg["phase"], gen_ch=cfg["gen_ch"])}',
        f'REG_WR w_env imm #0',
        f'REG_WR w_length imm #{self.us2cycles(cfg["pulse_len"], gen_ch=cfg["gen_ch"])}',
        f'REG_WR w_conf imm #9',
        f'REG_WR w_freq imm #{self.freq2reg(cfg["freq"], gen_ch=cfg["gen_ch"])}',
        'WMEM_WR [&0]'
    ]
    pstr2asm_inst(self, pstr_lst)

def SETUP_FEED(self):
    pstr_lst = [
        f'REG_WR w_gain imm #{str(15000)}',
        f'REG_WR w_phase imm #{self.deg2reg(90, gen_ch=2)}',
        f'REG_WR w_env imm #0',
        f'REG_WR w_length imm #{self.us2cycles(0.1, gen_ch=2)}',
        f'REG_WR w_conf imm #9',
        f'REG_WR w_freq imm #{self.freq2reg(1000, gen_ch=2)}',
        'WMEM_WR [&1]'
    ]
    pstr2asm_inst(self, pstr_lst)

def LFSR_sub(self):
    pstr_lst = [
        'LFSR:',
        'REG_WR r26 op -op(s1)',
        'REG_WR r28 op -op(MSH r26)',
        'RET',
    ]
    pstr2asm_inst(self, pstr_lst)

def set_wport_on(self):
    pstr_lst = ['WPON:', 'REG_WR w_gain imm #32766', 'WMEM_WR [&0]', 'RET']
    pstr2asm_inst(self, pstr_lst)

def set_wport_off(self):
    pstr_lst = ['WPOFF:', 'REG_WR w_gain imm #0', 'WMEM_WR [&0]', 'RET']
    pstr2asm_inst(self, pstr_lst)

def cond_out(self):
    # External CALL targets (LFSR, WPON, WPOFF) are not in this string's label_dict,
    # so they must be injected via self.asm_inst to be resolved at link time.
    pstr2asm_inst(self, ['CONDOUT:'])
    CALL(self, 'LFSR')
    pstr2asm_inst(self, ['REG_WR r28 op -op(r28 SR #5)', 'TEST -op(r28 AND #8) -uf'])
    CALL(self, 'WPON', cond='NZ')
    CALL(self, 'WPOFF', cond='Z')
    pstr2asm_inst(self, ['TEST -op(r28) -uf', 'RET'])

def rand_amp(self):
    pstr2asm_inst(self, ['RANDAMP:'])
    CALL(self, 'LFSR')
    pstr2asm_inst(self, ['REG_WR w_gain op -op(ABS r28)', 'WMEM_WR [&0]', 'RET'])

def read_and_jump(self):
    pstr2asm_inst(self, ['THRES:', 'REG_WR r31 op -op(r1)', 'TEST -op(r31 - #500) -uf'])
    CALL(self, 'playPULSE2', cond='Z')
    pstr2asm_inst(self, ['RET'])

def playPULSE1(self, port): 
    pstr2asm_inst(self, ['playPULSE1:', f'WPORT_WR p{port} wmem [&0]', 'RET'])

def playPULSE2(self, port): 
    pstr2asm_inst(self, ['playPULSE2:', f'WPORT_WR p{port} wmem [&1]', 'RET'])

In [33]:
class feedback_test(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        shots = cfg['shots']
        gain = cfg['gain']
        feed_ch = cfg['feed_ch']

        self.declare_gen(ch=gen_ch)
        self.declare_gen(ch=feed_ch)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'], freq=cfg['freq'], gen_ch=gen_ch)

        self.add_pulse(ch=gen_ch, name="mypulse", freq=cfg['freq'], phase=0,
                       length=cfg['pulse_len'], style="const", mode="oneshot",
                       stdysel="zero", gain=gain)
        
        self.add_pulse(ch=feed_ch, name="feedpulse", freq=1000, phase=0,
                       length=0.5, style="const", mode="oneshot",
                       stdysel="zero", gain=gain)

        cfg['gain'] = 32766
        
        SETUP_PULSE(self, cfg)
        
        # FIXED: Tell it to jump to the renamed label
        JUMP(self, "END_SETUP")
        
        playPULSE1(self, gen_ch)
        playPULSE2(self, feed_ch)

        fb_labels = [LFSR_sub, set_wport_on, set_wport_off, cond_out, read_and_jump, rand_amp]
        for label in fb_labels:
            label(self)
        
        NOP(self) # This now drops the 'END_SETUP:' label
        self.add_loop("ShotLoop", shots)

    def _body(self, cfg):
        CALL(self, 'RANDAMP') 
        CALL(self, 'playPULSE1')
        self.trigger(ros=[cfg['ro_ch']], t=cfg['trig_time'],  ddr4=True)
        self.wait_auto(cfg['feed_delay'])
        self.delay_auto(cfg['feed_delay'] + cfg['extra_delay'])
        self.trigger(ros=[cfg['ro_ch']], t=cfg['trig_time'],  ddr4=True)
        self.read_and_jump(cfg['ro_ch'], component='Q', threshold=500, test="<", label='NOPULSE')
        self.pulse(ch=cfg['feed_ch'], name="feedpulse", t=0)
        self.label("NOPULSE")
        self.delay_auto(t=cfg['meas_delay'] + cfg['pulse_len'])


# =============================================================================
# 3. Emulator Execution & Generation
# =============================================================================

config = {
    'gen_ch': 0, 'feed_ch': 2, 'ro_ch': 4, 'freq': 100, 'phase': 0,
    'trig_time': 0.01, 'ro_len': 1, 'shots': 10000, 'meas_delay': 10,
    'pulse_len': 1, 'gain': 1, "feed_delay": 0.2, 'extra_delay': 0.0,
}

prog = feedback_test(soccfg, reps=1, final_delay=.5, cfg=config)

dac_model_dir = REPO_ROOT / "models" / "Dac"
mem_directory = REPO_ROOT / "emulator"

# Make the mocked System-on-Chip (SOC)
soc_emu = emu.make_soc(memdir=mem_directory)
soc_emu.tproc.set_lfsr_cfg(2)  # mode 2: step on s1 read

# Generate and export the .mem files and AXI playback script
prep_results = emu.prepare(prog, soc_emu, memdir=mem_directory)
print(f"Memory artifacts successfully generated at: {prep_results['memdir']}")

Memory artifacts successfully generated at: /Users/sbf/Desktop/to_verilate/emulator/emulator


In [10]:
'''
Print program out to show what is going on
'''
print(prog)

macros:
	WriteReg(dst='s_core_w1', src=0)
	AsmInst(inst={'P_ADDR': 0, 'CMD': 'NOP'}, addr_inc=1)
	AsmInst(inst={'P_ADDR': 1, 'LINE': 1, 'LIT': '#32766', 'CMD': 'REG_WR', 'DST': 'w3', 'SRC': 'imm'}, addr_inc=1)
	AsmInst(inst={'P_ADDR': 2, 'LINE': 2, 'LIT': '#0', 'CMD': 'REG_WR', 'DST': 'w1', 'SRC': 'imm'}, addr_inc=1)
	AsmInst(inst={'P_ADDR': 3, 'LINE': 3, 'LIT': '#0', 'CMD': 'REG_WR', 'DST': 'w2', 'SRC': 'imm'}, addr_inc=1)
	AsmInst(inst={'P_ADDR': 4, 'LINE': 4, 'LIT': '#430', 'CMD': 'REG_WR', 'DST': 'w4', 'SRC': 'imm'}, addr_inc=1)
	AsmInst(inst={'P_ADDR': 5, 'LINE': 5, 'LIT': '#9', 'CMD': 'REG_WR', 'DST': 'w5', 'SRC': 'imm'}, addr_inc=1)
	AsmInst(inst={'P_ADDR': 6, 'LINE': 6, 'LIT': '#44810940', 'CMD': 'REG_WR', 'DST': 'w0', 'SRC': 'imm'}, addr_inc=1)
	AsmInst(inst={'P_ADDR': 7, 'LINE': 7, 'CMD': 'WMEM_WR', 'DST': '[&0]'}, addr_inc=1)
	AsmInst(inst={'P_ADDR': 0, 'CMD': 'NOP'}, addr_inc=1)
	AsmInst(inst={'P_ADDR': 1, 'LINE': 1, 'CMD': 'JUMP', 'LABEL': 'NOP'}, addr_inc=1)
	Label(label=